In [1]:
import pandas as pd
import os
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, Border, Side, PatternFill
from openpyxl.utils.dataframe import dataframe_to_rows

# 기본 스타일 설정
default_font = Font(name='Arial', size=11)
default_alignment = Alignment(horizontal='general', vertical='bottom')
default_border = Border(left=Side(border_style='thin'), right=Side(border_style='thin'),
                        top=Side(border_style='thin'), bottom=Side(border_style='thin'))
default_fill = PatternFill(fill_type=None)

# 1. 병합 대상 파일 목록 (현재는 세 파일로 고정)
excel_files = [
    "datalab (78).xlsx",
    "datalab (79).xlsx",
    "datalab (80).xlsx"
]

# 2. URL 및 키워드 정보 초기화
df_csv = pd.DataFrame(columns=["그룹번호", "URL"])
merged_df = pd.DataFrame()
first_dates = None

# 3. 병합 루프
for idx, file in enumerate(excel_files):
    # URL 추출 (B1 셀)
    url = pd.read_excel(file, nrows=1, header=None, usecols="B").iloc[0, 0]
    df_csv.loc[idx, "그룹번호"] = idx + 1
    df_csv.loc[idx, "URL"] = url

    # 메뉴 데이터 읽기 (7번째 줄부터)
    df = pd.read_excel(file, header=6, nrows=26)

    # 날짜 열 저장
    if first_dates is None:
        first_dates = df.iloc[:, 0].copy()

    # 짝수 열만 추출 (메뉴 항목들)
    even_cols = df.columns[1::2]
    df = df[even_cols]

    # 병합
    merged_df = pd.concat([merged_df, df], axis=1)

# 4. 월 열 추가
merged_df.insert(0, "월", pd.to_datetime(first_dates).dt.strftime('%Y-%m'))

# 5. 병합 결과 저장
output_path = "병합된_상세메뉴_월별추세_결과.xlsx"
wb = Workbook()

# 월별추세 시트 생성
ws1 = wb.active
ws1.title = "월별추세"
for r in dataframe_to_rows(merged_df, index=False, header=True):
    ws1.append(r)

# 키워드+URL 시트 생성
ws2 = wb.create_sheet("키워드+URL")
for r in dataframe_to_rows(df_csv, index=False, header=True):
    ws2.append(r)

# 스타일 적용
for ws in [ws1, ws2]:
    for row in ws.iter_rows(min_row=1, max_row=ws.max_row, min_col=1, max_col=ws.max_column):
        for cell in row:
            cell.font = default_font
            cell.alignment = default_alignment
            cell.border = default_border
            cell.fill = default_fill

# 저장
wb.save(output_path)
print(f"병합 완료: {output_path}")

C:\Users\Admin\anaconda3\lib\site-packages\openpyxl\styles\stylesheet.py:226: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
C:\Users\Admin\anaconda3\lib\site-packages\openpyxl\styles\stylesheet.py:226: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
C:\Users\Admin\anaconda3\lib\site-packages\openpyxl\styles\stylesheet.py:226: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
C:\Users\Admin\anaconda3\lib\site-packages\openpyxl\styles\stylesheet.py:226: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
C:\Users\Admin\anaconda3\lib\site-packages\openpyxl\styles\stylesheet.py:226: UserWarning: Workbook contains

병합 완료: 병합된_상세메뉴_월별추세_결과.xlsx
